# Data Understanding

In [22]:
!pip install imbalanced-learn


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE


In [24]:
df = pd.read_csv('data.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [26]:
df.shape

(7043, 21)

# Data Preprocessing

## Global Cleaning

###  Duplicate Value

In [27]:
df.duplicated().sum()

0

### Handling Data Type

*'Total Charges'* has Object datatype. *'Total Charges'* must be convert to numerical datatype

In [28]:
# Ubah string kosong/spasi menjadi NaN (Not a Number)
df['TotalCharges'] = df['TotalCharges'].replace(" ", pd.NA)

# Konversi kolom menjadi tipe numeric (float)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# Tangani nilai NaN (karena pelanggan baru dengan tenure=0 belum membayar, diisi dengan 0)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Verifikasi perubahan tipe data
print(df['TotalCharges'].dtype)
df.info()

float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   

In [29]:
df[df['TotalCharges'] == 0].head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,0.0,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,0.0,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,0.0,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,0.0,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,0.0,No


### Drop 'CustomerID' 

In [30]:
df = df.drop(columns=['customerID'])
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Penyederhanaan Category

In [31]:
# Penyederhanaan Kategori (No internet service / No phone service -> No)
replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace({'No internet service': 'No'})

df['MultipleLines'] = df['MultipleLines'].replace({'No phone service': 'No'})

### Labeling Target Columns

In [32]:
# Convert Target Variable 'Churn' (Yes/No -> 1/0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Global Cleaning Selesai. Shape data:", df.shape)

Global Cleaning Selesai. Shape data: (7043, 20)


## Train - Test Split Dataset

In [33]:
X = df.drop(columns=['Churn'])
y = df['Churn']

In [34]:
# Split Data 80% Train, 20% Test dengan stratify agar proporsi kelas awal tetap terjaga
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

In [35]:
print(f"\n[Split Data 80:20]")
print(f"Data Train original shape: {X_train.shape}")
print(f"Data Test original shape : {X_test.shape}")


[Split Data 80:20]
Data Train original shape: (5634, 19)
Data Test original shape : (1409, 19)


## Preprocessing After Split Data

In [21]:
# Loop untuk memeriksa setiap kolom pada dataframe df
for col in df.columns:
    print(col)
    print(f"  Tipe Data : {df[col].dtype}")
    print(f"  Nilai Unik: {df[col].unique()}")
    print("-" * 50)  # Garis pembatas antar kolom


gender
  Tipe Data : object
  Nilai Unik: ['Female' 'Male']
--------------------------------------------------
SeniorCitizen
  Tipe Data : int64
  Nilai Unik: [0 1]
--------------------------------------------------
Partner
  Tipe Data : object
  Nilai Unik: ['Yes' 'No']
--------------------------------------------------
Dependents
  Tipe Data : object
  Nilai Unik: ['No' 'Yes']
--------------------------------------------------
tenure
  Tipe Data : int64
  Nilai Unik: [ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]
--------------------------------------------------
PhoneService
  Tipe Data : object
  Nilai Unik: ['No' 'Yes']
--------------------------------------------------
MultipleLines
  Tipe Data : object
  Nilai Unik: ['No' 'Yes']
--------------------------------------------------
InternetService
  Tipe Dat

In [36]:
# Identifikasi Fitur Numerik dan Kategorikal
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_features = [col for col in X_train.columns if col not in num_features]

### Pipeline Scaling & Encoding

In [37]:
# Pipeline Scaling & Encoding
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Encoding Data
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ]
)

Pengaturan di dalam OneHotEncoder artinya:
1.  `drop='first'`: Menghapus kolom tiruan (dummy) pertama untuk mencegah multicollinearity (hubungan antar variabel yang terlalu kuat/redundan). Contoh: jika ada kategori Gender (Male/Female), cukup dibuat satu kolom Is_Male (1 jika Male, 0 jika Female).
2.  `sparse_output=False`: Memaksa hasil transformasi berupa matriks padat biasa (dense array) supaya mudah dibaca, bukan matriks renggang (sparse matrix).
3. ` handle_unknown='ignore'`: Jika nanti ada kategori baru di data uji yang belum pernah dilihat saat latihan, komputer tidak akan eror. Komputer hanya akan memberi nilai 0 pada semua kolom baru tersebut.

ColumnTransformer bertugas memetakan rumus pengolahan yang berbeda ke kolom yang tepat secara bersamaan:
1.  `Jalur Numerik ('num')`: Menerapkan modul numeric_transformer (biasanya berisi pengisian data kosong atau scaling) khusus untuk kolom-kolom yang terdaftar di num_features.
2.  `Jalur Kategori ('cat')`: Menerapkan modul categorical_transformer (OneHotEncoder di atas) khusus untuk kolom-kolom yang terdaftar di cat_features.

### Fit Data Train

In [39]:
# Fit HANYA pada X_train, Transform pada X_train & X_test
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

#### 1. preprocessor.fit_transform(X_train) (Belajar + Ubah)
Perintah ini melakukan dua aksi sekaligus pada data latihan:
*  `fit (Belajar)`: Komputer akan mempelajari karakteristik data X_train. Contohnya, mencari nilai rata-rata untuk mengisi data kosong (imputasi), mencari nilai minimum/maksimum untuk scaling, dan mencatat semua kategori unik yang ada pada kolom teks untuk One-Hot Encoding.
*  `transform (Ubah)`: Setelah selesai mempelajari rumusnya, komputer langsung mengubah data X_train asli menjadi format baru (angka) berdasarkan rumus yang sudah dipelajari tadi.

---

#### 2. preprocessor.transform(X_test) (HANYA Ubah)
Perintah ini tidak boleh belajar dari data uji, melainkan hanya menggunakan rumus yang sudah didapatkan dari X_train sebelumnya:
*  Komputer tidak mencari rata-rata atau kategori baru di dalam X_test.
*  Komputer langsung mengubah data X_test menggunakan standar, angka rata-rata, dan kategori yang sudah dikunci saat proses fit pada X_train tadi.

#### Mengapa X_test Tidak Boleh di-fit?
Bayangkan sedang ujian. Data X_train adalah buku pelajaran, sedangkan X_test adalah lembar soal ujian asli. Jika melakukan fit pada X_test, itu sama saja dengan mengintip kunci jawaban atau bocoran soal sebelum ujian dimulai. Akibatnya, nilai evaluasi model akan terlihat sangat bagus di komputer (overfitting), tetapi akan langsung gagal total saat menghadapi data rias yang baru di dunia nyata.Setelah data diubah menjadi X_train_preprocessed dan X_test_preprocessed, langkah selanjutnya adalah memasukkannya ke dalam algoritma model. Apakah ingin membuat model klasifikasi (seperti memprediksi Churn/Tidak Churn) menggunakan data yang sudah bersih ini?

In [40]:
# Mengambil kembali nama kolom hasil One-Hot Encoding
cat_encoder = preprocessor.named_transformers_['cat']['onehot']
encoded_cat_cols = cat_encoder.get_feature_names_out(cat_features)
all_feature_names = num_features + list(encoded_cat_cols)

In [41]:

# Konversi ke DataFrame
X_train_ready = pd.DataFrame(X_train_preprocessed, columns=all_feature_names)
X_test_ready = pd.DataFrame(X_test_preprocessed, columns=all_feature_names)